# 🌾 Crop Recommendation — New Dataset (Categorical + Range Features)

This adapts the same pipeline (baseline + strong models, macro-F1-weighted composite scoring, training/validation gap check, confusion matrix, feature importance) to the new dataset schema:

- **Target:** `CROPS`
- **Categorical features:** `TYPE_OF_CROP`, `SOIL`, `SEASON`, `SOWN`, `HARVESTED`, `WATER_SOURCE`
- **Numeric (min/max range) features:** `SOIL_PH`/`SOIL_PH_HIGH`, `CROPDURATION`/`CROPDURATION_MAX`, `TEMP`/`MAX_TEMP`, `WATERREQUIRED`/`WATERREQUIRED_MAX`, `RELATIVE_HUMIDITY`/`RELATIVE_HUMIDITY_MAX`, `N`/`N_MAX`, `P`/`P_MAX`, `K`/`K_MAX`

⚠️ **Assumption:** the CSV is loaded as `"Crop_Dataset.csv"` in Cell 2 — rename your file to match, or edit that one line to your actual filename/path.

Since categorical columns now exist, preprocessing uses a `ColumnTransformer` (`StandardScaler` for numeric + `OneHotEncoder` for categorical) wrapped inside each model's `Pipeline`, so scaling/encoding is always fit on training data only and just applied (`.transform()`) to test/new data.

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import time
import pickle
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier

## Cell 2 — Load Dataset

In [2]:
crop = pd.read_csv("Crop_Dataset.csv")

print("Dataset Shape:", crop.shape)

crop.head()

Dataset Shape: (57000, 23)


,CROPS,TYPE_OF_CROP,SOIL,SEASON,SOWN,HARVESTED,WATER_SOURCE,SOIL_PH,SOIL_PH_HIGH,CROPDURATION,...,WATERREQUIRED,WATERREQUIRED_MAX,RELATIVE_HUMIDITY,RELATIVE_HUMIDITY_MAX,N,N_MAX,P,P_MAX,K,K_MAX
0,rice,cereals,Alluvial soil,kharif,Jun,Sep,irrigated,7.6,8.0,116.9,...,2462.3,2500,73.8,80,82.4,100,40.7,60,42.2,60
1,rice,cereals,Loamy soil,kharif,Jul,Oct,rainfed,6.2,8.0,117.9,...,1237.5,2500,60.9,80,90.5,100,51.3,60,46.2,60
2,rice,cereals,Clay soil,kharif,Jun,Sep,irrigated,6.7,8.0,117.7,...,1075.1,2500,67.5,80,86.2,100,50.7,60,44.4,60
3,rice,cereals,Alluvial soil,kharif,Jul,Oct,rainfed,6.1,8.0,149.8,...,1549.9,2500,73.6,80,91.3,100,51.3,60,44.5,60
4,rice,cereals,Loamy soil,kharif,Jun,Sep,irrigated,8.0,8.0,131.7,...,1306.4,2500,60.3,80,81.3,100,48.6,60,51.0,60


## Cell 3 — Basic Dataset Check

In [3]:
print("Dataset Information:")
crop.info()

print("\nMissing Values:")
print(crop.isnull().sum())

print("\nDuplicate Rows:")
print(crop.duplicated().sum())

print("\nNumber of Crop Classes:")
print(crop["CROPS"].nunique())

print("\nClass Distribution:")
print(crop["CROPS"].value_counts())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57000 entries, 0 to 56999
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CROPS                  57000 non-null  object 
 1   TYPE_OF_CROP           57000 non-null  object 
 2   SOIL                   57000 non-null  object 
 3   SEASON                 57000 non-null  object 
 4   SOWN                   57000 non-null  object 
 5   HARVESTED              57000 non-null  object 
 6   WATER_SOURCE           57000 non-null  object 
 7   SOIL_PH                57000 non-null  float64
 8   SOIL_PH_HIGH           57000 non-null  float64
 9   CROPDURATION           57000 non-null  float64
 10  CROPDURATION_MAX       57000 non-null  int64  
 11  TEMP                   57000 non-null  float64
 12  MAX_TEMP               57000 non-null  int64  
 13  WATERREQUIRED          57000 non-null  float64
 14  WATERREQUIRED_MAX      57000 non-

## Cell 4 — Define Feature Groups
Categorical columns go to one-hot encoding, numeric (including all the `_MAX`/`_HIGH` range columns) go to scaling.

In [4]:
categorical_features = [
    "TYPE_OF_CROP",
    "SOIL",
    "SEASON",
    "SOWN",
    "HARVESTED",
    "WATER_SOURCE"
]

numeric_features = [
    "SOIL_PH",
    "SOIL_PH_HIGH",
    "CROPDURATION",
    "CROPDURATION_MAX",
    "TEMP",
    "MAX_TEMP",
    "WATERREQUIRED",
    "WATERREQUIRED_MAX",
    "RELATIVE_HUMIDITY",
    "RELATIVE_HUMIDITY_MAX",
    "N",
    "N_MAX",
    "P",
    "P_MAX",
    "K",
    "K_MAX"
]

all_features = numeric_features + categorical_features

X = crop[all_features]
y = crop["CROPS"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nCategorical unique values:")
for col in categorical_features:
    print(f"{col}: {crop[col].nunique()} unique ->", crop[col].unique())

X Shape: (57000, 22)
y Shape: (57000,)

Categorical unique values:
TYPE_OF_CROP: 10 unique -> ['cereals' 'millets' 'pulses' 'oil seeds' 'fibre crop' 'sugar crops'
 'vegetables' 'colecrops' 'Root&tuber' 'bulbvegetables']
SOIL: 34 unique -> ['Alluvial soil' 'Loamy soil' 'Clay soil' 'well-drained soil' 'Red soil'
 'clay Loamy soil' 'sandy loamy\xa0soil' 'Black Soil' 'Sandy soil'
 'shallow Black Soil' 'sandy Loamy soil' 'black cotton soil'
 'Sandy\xa0soil' '\xa0cotton\xa0soil' 'Sandy soil\xa0' 'medium Black Soil'
 'heavy Black Soil' '\xa0light soi' 'heavy soil\xa0' 'deep soil'
 'loamy\xa0soil' 'sandy clay Loamy soil\xa0' 'silty Loamy soil'
 'salty clay Loamy soil' 'red Loamy soil' 'brown Loamy soil'
 'clay Loamy soil ' 'Laterite soil' 'well-drained loamy\xa0soil'
 'light Loamy soil' 'friable soil' 'well-grained deep loamy moist soil'
 'red lateritic Loamy soil' 'rich red Loamy soil']
SEASON: 3 unique -> ['kharif' 'rabi' 'Zaid']
SOWN: 8 unique -> ['Jun' 'Jul' 'Oct' 'Nov' 'Dec' 'Mar' 'Apr' '

## Cell 5 — Encode Target

In [5]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Crop Classes:")

for i, crop_name in enumerate(label_encoder.classes_):
    print(i, "=", crop_name)

Crop Classes:
0 = Cabbage
1 = Pearl millet
2 = annual moringa
3 = ash gourd
4 = beetroot
5 = bengalgram
6 = bhendi
7 = bitter gourd
8 = blackgram
9 = bottle gourd
10 = brinjal
11 = capsicum
12 = carrot
13 = castor
14 = cauliflower
15 = chillies
16 = chowchow
17 = cluster bean
18 = cotton
19 = cowpea
20 = cucumber
21 = elephant foot yam
22 = french bean
23 = gingely
24 = greengram
25 = groundnut
26 = horsegram
27 = jute
28 = kudiraivali
29 = maize
30 = muskmelon
31 = onion
32 = panivaragu
33 = peas
34 = pumpkin
35 = radish
36 = ragi
37 = redgram
38 = ribbed gourd
39 = rice
40 = samai
41 = small onion
42 = snake gourd
43 = sorghum
44 = soyabean
45 = sugarbeet
46 = sugarcane
47 = sunflower
48 = sweet potato
49 = tapoica
50 = thinai
51 = tinda
52 = tomato
53 = varagu
54 = vegetable cowpea
55 = watermelon
56 = wheat


## Cell 6 — Train/Test Split
Stratified so every crop class stays balanced across train and test.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (45600, 22)
Testing Data: (11400, 22)


## Cell 7 — Preprocessor
`ColumnTransformer` scales numeric columns and one-hot encodes categorical columns. `handle_unknown="ignore"` prevents a crash if a new/unseen category shows up at prediction time.

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    sparse_threshold=0
)

## Cell 8 — Baseline Models
Each wrapped with the shared `preprocessor` so encoding/scaling is fit only on training data.

In [8]:
baseline_models = {

    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]),

    "Gaussian Naive Bayes": Pipeline([
        ("preprocessor", preprocessor),
        ("model", GaussianNB())
    ]),

    "KNN": Pipeline([
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier())
    ]),

    "SVM": Pipeline([
        ("preprocessor", preprocessor),
        ("model", SVC())
    ]),

    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ])
}

## Cell 9 — Strong Models

In [9]:
strong_models = {

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]),

    "Extra Trees": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            ExtraTreesClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]),

    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingClassifier(
                random_state=42
            )
        )
    ]),

    "XGBoost": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="multi:softprob",
                num_class=len(label_encoder.classes_),
                eval_metric="mlogloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ])
}

## Cell 10 — Combine Models

In [10]:
models = {}

models.update(baseline_models)
models.update(strong_models)

print("Total Models:", len(models))

for name in models:
    print(name)

Total Models: 9
Logistic Regression
Gaussian Naive Bayes
KNN
SVM
Decision Tree
Random Forest
Extra Trees
Gradient Boosting
XGBoost


## Cell 11 — Evaluation Function
Same metrics as before: macro precision/recall/F1, accuracy, 5-fold CV (macro F1), training/validation gap, confusion matrix, timing.

In [11]:
def evaluate_model(name, model):

    # =====================================================
    # TRAINING
    # =====================================================

    start_train = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    end_train = time.perf_counter()

    training_time = (
        end_train - start_train
    )


    # =====================================================
    # TRAINING PREDICTION
    # =====================================================

    y_train_pred = model.predict(
        X_train
    )

    train_accuracy = accuracy_score(
        y_train,
        y_train_pred
    )


    # =====================================================
    # VALIDATION / TEST PREDICTION
    # =====================================================

    start_predict = time.perf_counter()

    y_test_pred = model.predict(
        X_test
    )

    end_predict = time.perf_counter()

    inference_time = (
        end_predict - start_predict
    )


    # =====================================================
    # TEST METRICS
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_test_pred
    )

    precision_macro = precision_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_test,
        y_test_pred,
        average="macro",
        zero_division=0
    )


    # =====================================================
    # TRAINING / VALIDATION GAP
    # =====================================================

    training_validation_gap = (
        train_accuracy - accuracy
    )


    # =====================================================
    # 5-FOLD STRATIFIED CROSS VALIDATION
    # =====================================================

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    cv_mean = cv_scores.mean()

    cv_std = cv_scores.std()


    # =====================================================
    # CONFUSION MATRIX
    # =====================================================

    cm = confusion_matrix(
        y_test,
        y_test_pred
    )


    # =====================================================
    # RETURN ALL RESULTS
    # =====================================================

    return {

        "Model": name,

        "F1 Macro": f1_macro,

        "Accuracy": accuracy,

        "Precision Macro": precision_macro,

        "Recall Macro": recall_macro,

        "CV Mean": cv_mean,

        "CV Std": cv_std,

        "Training Accuracy": train_accuracy,

        "Training Validation Gap":
            training_validation_gap,

        "Training Time":
            training_time,

        "Inference Time":
            inference_time,

        "Confusion Matrix":
            cm,

        "Predictions":
            y_test_pred,

        "Model Object":
            model
    }

## Cell 12 — Train All Models

In [13]:
results = []

for name, model in models.items():

    print(
        "Training:",
        name
    )

    result = evaluate_model(
        name,
        model
    )

    results.append(
        result
    )

print("\nAll models completed.")

Training: Logistic Regression
Training: Gaussian Naive Bayes
Training: KNN


KeyboardInterrupt: 

## Cell 13 — Create Result Table

In [ ]:
results_df = pd.DataFrame([

    {
        "Model": r["Model"],

        "F1 Macro": r["F1 Macro"],

        "Accuracy": r["Accuracy"],

        "Precision Macro":
            r["Precision Macro"],

        "Recall Macro":
            r["Recall Macro"],

        "CV Mean":
            r["CV Mean"],

        "CV Std":
            r["CV Std"],

        "Training Accuracy":
            r["Training Accuracy"],

        "Training Validation Gap":
            r["Training Validation Gap"],

        "Training Time":
            r["Training Time"],

        "Inference Time":
            r["Inference Time"]
    }

    for r in results
])

results_df

## Cell 14 — ⭐ Composite Score

Weights: F1 Macro 30%, Accuracy 20%, Precision Macro 15%, Recall Macro 15%, CV Mean 20%.

In [ ]:
results_df["Composite Score"] = (

    0.30 * results_df["F1 Macro"]

    +

    0.20 * results_df["Accuracy"]

    +

    0.15 * results_df["Precision Macro"]

    +

    0.15 * results_df["Recall Macro"]

    +

    0.20 * results_df["CV Mean"]
)

## Cell 15 — Gap Penalty & Stability Score

In [ ]:
results_df["Gap Penalty"] = (
    results_df["Training Validation Gap"]
    .abs()
)

results_df["Stability Score"] = (
    1 -
    results_df["Gap Penalty"]
)

## Cell 16 — Final Score
90% Composite Score + 10% Stability Score.

In [ ]:
results_df["Final Score"] = (

    0.90 * results_df["Composite Score"]

    +

    0.10 * results_df["Stability Score"]
)

## Cell 17 — Rank Models

In [ ]:
results_df = results_df.sort_values(
    by="Final Score",
    ascending=False
)

results_df = results_df.reset_index(
    drop=True
)

results_df["Rank"] = (
    results_df.index + 1
)

results_df[
    [
        "Rank",
        "Model",
        "F1 Macro",
        "Accuracy",
        "Precision Macro",
        "Recall Macro",
        "CV Mean",
        "CV Std",
        "Training Accuracy",
        "Training Validation Gap",
        "Composite Score",
        "Stability Score",
        "Final Score"
    ]
]

## Cell 18 — Percentage Result

In [ ]:
display_df = results_df.copy()

percentage_columns = [
    "F1 Macro",
    "Accuracy",
    "Precision Macro",
    "Recall Macro",
    "CV Mean",
    "Composite Score",
    "Stability Score",
    "Final Score"
]

for col in percentage_columns:

    display_df[col] = (
        display_df[col] * 100
    ).round(2)

display_df[
    [
        "Rank",
        "Model",
        "F1 Macro",
        "Accuracy",
        "Precision Macro",
        "Recall Macro",
        "CV Mean",
        "CV Std",
        "Training Accuracy",
        "Training Validation Gap",
        "Composite Score",
        "Stability Score",
        "Final Score"
    ]
]

## Cell 19 — 🏆 Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]

best_result = next(
    r for r in results
    if r["Model"] == best_model_name
)

best_model = best_result["Model Object"]

print("=" * 60)

print("🏆 BEST MODEL")

print("=" * 60)

print(
    "Model:",
    best_model_name
)

print(
    f"F1 Macro: "
    f"{results_df.iloc[0]['F1 Macro'] * 100:.2f}%"
)

print(
    f"Accuracy: "
    f"{results_df.iloc[0]['Accuracy'] * 100:.2f}%"
)

print(
    f"Precision Macro: "
    f"{results_df.iloc[0]['Precision Macro'] * 100:.2f}%"
)

print(
    f"Recall Macro: "
    f"{results_df.iloc[0]['Recall Macro'] * 100:.2f}%"
)

print(
    f"CV Score: "
    f"{results_df.iloc[0]['CV Mean'] * 100:.2f}%"
)

print(
    f"Training/Validation Gap: "
    f"{results_df.iloc[0]['Training Validation Gap'] * 100:.2f}%"
)

print(
    f"Final Score: "
    f"{results_df.iloc[0]['Final Score'] * 100:.2f}%"
)

print("=" * 60)

## Cell 20 — F1 Macro Comparison

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=results_df,
    x="F1 Macro",
    y="Model"
)

plt.title("Macro F1-Score Comparison")
plt.xlabel("Macro F1-Score")
plt.ylabel("Model")
plt.xlim(0, 1)

plt.show()

## Cell 21 — Accuracy Comparison

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=results_df,
    x="Accuracy",
    y="Model"
)

plt.title("Accuracy Comparison")
plt.xlabel("Accuracy")
plt.ylabel("Model")
plt.xlim(0, 1)

plt.show()

## Cell 22 — Precision Macro

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=results_df,
    x="Precision Macro",
    y="Model"
)

plt.title("Macro Precision Comparison")
plt.xlabel("Macro Precision")
plt.ylabel("Model")
plt.xlim(0, 1)

plt.show()

## Cell 23 — Recall Macro

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=results_df,
    x="Recall Macro",
    y="Model"
)

plt.title("Macro Recall Comparison")
plt.xlabel("Macro Recall")
plt.ylabel("Model")
plt.xlim(0, 1)

plt.show()

## Cell 24 — Cross Validation

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=results_df,
    x="CV Mean",
    y="Model"
)

plt.title("5-Fold Cross Validation Macro F1")
plt.xlabel("CV Mean Macro F1")
plt.ylabel("Model")
plt.xlim(0, 1)

plt.show()

## Cell 25 — Final Score

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=results_df,
    x="Final Score",
    y="Model"
)

plt.title("Final Model Selection Score")
plt.xlabel("Final Score")
plt.ylabel("Model")
plt.xlim(0, 1)

plt.show()

## Cell 26 — Training/Validation Gap

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=results_df,
    x="Training Validation Gap",
    y="Model"
)

plt.title("Training / Validation Gap")
plt.xlabel("Training Accuracy - Validation Accuracy")
plt.ylabel("Model")

plt.show()

## Cell 27 — Best Model Classification Report

In [ ]:
best_predictions = best_result[
    "Predictions"
]

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

## Cell 28 — 🟦 Confusion Matrix

In [ ]:
cm = best_result[
    "Confusion Matrix"
]

plt.figure(figsize=(16, 13))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()

plt.show()

## Cell 29 — Per-Class F1 Score

In [ ]:
report = classification_report(
    y_test,
    best_predictions,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0
)

class_f1 = pd.DataFrame({

    "Crop":
        label_encoder.classes_,

    "F1 Score": [
        report[c]["f1-score"]
        for c in label_encoder.classes_
    ]
})

class_f1 = class_f1.sort_values(
    by="F1 Score"
)

class_f1

## Cell 30 — Per-Class F1 Graph

In [ ]:
plt.figure(figsize=(12, 8))

sns.barplot(
    data=class_f1,
    x="F1 Score",
    y="Crop"
)

plt.title(f"Per-Class F1 Score - {best_model_name}")
plt.xlabel("F1 Score")
plt.ylabel("Crop")
plt.xlim(0, 1)

plt.show()

## Cell 31 — Feature Importance
After one-hot encoding, feature names come from the fitted `ColumnTransformer` (`get_feature_names_out()`), not the original `all_features` list.

In [ ]:
fitted_preprocessor = best_model.named_steps["preprocessor"]
model_inside_pipeline = best_model.named_steps["model"]

encoded_feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(
    model_inside_pipeline,
    "feature_importances_"
):

    importance_df = pd.DataFrame({

        "Feature": encoded_feature_names,

        "Importance": model_inside_pipeline.feature_importances_

    })

    importance_df = importance_df.sort_values(
        by="Importance",
        ascending=False
    ).head(20)

    display(importance_df)

    plt.figure(figsize=(10, 8))

    sns.barplot(
        data=importance_df,
        x="Importance",
        y="Feature"
    )

    plt.title(f"Top 20 Feature Importance - {best_model_name}")

    plt.show()

else:

    print(
        "Feature importance is not directly "
        "available for this model."
    )

## Cell 32 — New Crop Prediction
Fill in a new sample using the same columns as `all_features` — numeric values plus the categorical fields (`TYPE_OF_CROP`, `SOIL`, `SEASON`, `SOWN`, `HARVESTED`, `WATER_SOURCE`). The pipeline's `ColumnTransformer` handles scaling/encoding automatically.

In [ ]:
# ============================================================
# NEW INPUT — edit these values for a real prediction
# ============================================================

new_sample = {
    "SOIL_PH": 6.8,
    "SOIL_PH_HIGH": 8.0,
    "CROPDURATION": 120,
    "CROPDURATION_MAX": 150,
    "TEMP": 27.5,
    "MAX_TEMP": 40,
    "WATERREQUIRED": 1500,
    "WATERREQUIRED_MAX": 2500,
    "RELATIVE_HUMIDITY": 68,
    "RELATIVE_HUMIDITY_MAX": 80,
    "N": 85,
    "N_MAX": 100,
    "P": 45,
    "P_MAX": 60,
    "K": 45,
    "K_MAX": 60,
    "TYPE_OF_CROP": "cereals",
    "SOIL": "Alluvial soil",
    "SEASON": "kharif",
    "SOWN": "Jun",
    "HARVESTED": "Sep",
    "WATER_SOURCE": "irrigated"
}

new_data = pd.DataFrame(
    [new_sample],
    columns=all_features
)

prediction = best_model.predict(new_data)

predicted_crop = (
    label_encoder
    .inverse_transform(prediction)[0]
)

print("=" * 50)
print("🌱 Recommended Crop:", predicted_crop)
print("=" * 50)

## Cell 33 — Top 3 Recommendations

In [ ]:
if hasattr(best_model, "predict_proba"):

    probabilities = best_model.predict_proba(new_data)[0]

    top_indices = np.argsort(probabilities)[::-1][:3]

    print("=" * 50)
    print("🌱 TOP 3 CROP RECOMMENDATIONS")
    print("=" * 50)

    for rank, index in enumerate(top_indices, start=1):

        crop_name = label_encoder.classes_[index]
        probability = probabilities[index] * 100

        print(f"{rank}. {crop_name} → {probability:.2f}%")

else:

    print("Probability is not available for this model.")

## Cell 34 — Save Model

In [ ]:
with open("best_crop_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

with open("crop_label_encoder.pkl", "wb") as file:
    pickle.dump(label_encoder, file)

print("Best model saved successfully!")

## Cell 35 — Save Model Comparison

In [ ]:
results_df.to_csv(
    "model_comparison_results.csv",
    index=False
)

print("Results saved successfully!")

## Notes on this dataset vs. the earlier one

- The old dataset (`Crop_recommendation.csv`) had only 7 purely numeric features. This one adds categorical context (soil type, season, sowing/harvest month, water source) plus paired min/max ranges for pH, duration, temperature, water need, humidity, N, P, and K — all handled through the `ColumnTransformer` instead of a plain `StandardScaler`.
- Because `OneHotEncoder` expands each categorical column into multiple 0/1 columns, the feature-importance step reads names from `get_feature_names_out()` rather than the original column list, and only the top 20 are shown for readability.
- The Macro-F1-weighted composite scoring, training/validation gap check, and confusion-matrix analysis carry over unchanged from the previous notebook.